In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from pydantic import BaseModel, Field
from IPython.display import Image, display

In [ ]:
chat_template_country_1 = ChatPromptTemplate.from_template(
    '''
    Analyze {country_1} historical competitivness and squad depth.
    '''
)

chat_template_country_2 = ChatPromptTemplate.from_template(
    '''
    Analyze {country_2} historical competitivness and squad depth.
    '''
)

chat_template_winning_country = ChatPromptTemplate.from_template(
    '''
    I'm an intermediate-level football fan studying the history of the game.

    Consider the following countries: {country_1} and {country_2}

    Which team would be likely to win the World Cup?
    '''
)

In [ ]:
class CountryAnalysis(BaseModel):
    """Analysis of a national team's historical competitiveness and squad depth."""

    country: str = Field(description="The country being analyzed")
    historical_competitiveness: str = Field(
        description="Assessment of historical World Cup and tournament performance",
    )
    squad_depth: str = Field(
        description="Assessment of current and historical squad depth",
    )


class WorldCupPrediction(BaseModel):
    """Prediction of which team is more likely to win the World Cup."""

    likely_winner: str = Field(
        description="The country most likely to win the World Cup",
    )
    reasoning: str = Field(
        description="Brief explanation of the prediction",
    )

In [ ]:
llm = ChatLiteLLM(
    model="gpt-5.4-mini",
    temperature=0,
    model_kwargs={'seed': 365},
    max_tokens=200,
)

In [ ]:
llm_country_1 = llm.with_structured_output(CountryAnalysis)
llm_country_2 = llm.with_structured_output(CountryAnalysis)
llm_plan = llm.with_structured_output(WorldCupPrediction)

In [ ]:
chain_country_1 = chat_template_country_1 | llm_country_1
chain_country_2 = chat_template_country_2 | llm_country_2

In [ ]:
chain_parallel = RunnableParallel({'country_1': chain_country_1, 'country_2': chain_country_2})

In [ ]:
chain_parallel.invoke({'country_1': 'Brazil', 'country_2': 'Argentina'})

In [ ]:
chain_plan1 = (
    RunnableParallel({'country_1': chain_country_1, 'country_2': chain_country_2})
    | chat_template_winning_country
    | llm_plan
)

In [ ]:
chain_plan2 = (
    {'country_1': chain_country_1, 'country_2': chain_country_2}
    | chat_template_winning_country
    | llm_plan
)

In [ ]:
chain_plan2.invoke({'country_1': 'Brazil', 'country_2': 'Argentina'})

In [ ]:
display(Image(chain_plan2.get_graph().draw_mermaid_png()))